## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-03-11-51-00 +0000,wsj,U.S. Factory Activity Continues to Expand,https://www.wsj.com/logistics-report/u-s-facto...
1,2026-11-03-11-50-45 +0000,bbc,Flights from UK airports among three targeted ...,https://www.bbc.co.uk/news/articles/c6x2wd4v04...
2,2026-11-03-11-48-00 +0000,wsj,Apollo Global Swings to a Loss on Investment-R...,https://www.wsj.com/business/earnings/apollo-g...
3,2026-11-03-11-45-04 +0000,nyt,Election Day Could Be a Rainy One in Some Areas,https://www.nytimes.com/2026/11/03/us/politics...
4,2026-11-03-11-43-00 +0000,nyt,How Will the Midterms Play Out? These Races Ma...,https://www.nytimes.com/2026/11/03/us/midterm-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2926/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
273,trump,44
19,election,31
38,midterm,22
25,midterms,19
179,california,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
44,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...,177
291,2026-11-02-19-34-04 +0000,chitri,President Donald Trump spends day before midte...,https://www.chicagotribune.com/2026/11/02/dona...,157
148,2026-11-03-02-55-07 +0000,nypost,Trump says Republicans should ‘win in a landsl...,https://nypost.com/2026/11/02/us-news/trump-sa...,132
171,2026-11-03-00-43-43 +0000,nypost,"Xavier Becerra, Steve Hilton make final push a...",https://nypost.com/2026/11/02/us-news/xavier-b...,124
214,2026-11-02-22-40-00 +0000,wsj,Midterm candidates across the country are maki...,https://www.wsj.com/politics/policy/midterm-ca...,121


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
44,177,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
156,98,2026-11-03-02-00-00 +0000,wsj,Tuesday’s midterm elections will offer clues t...,https://www.wsj.com/politics/elections/what-th...
163,81,2026-11-03-01-18-57 +0000,nypost,8 California House races to watch: GOP vs. GOP...,https://nypost.com/2026/11/02/us-news/8-califo...
214,70,2026-11-02-22-40-00 +0000,wsj,Midterm candidates across the country are maki...,https://www.wsj.com/politics/policy/midterm-ca...
81,61,2026-11-03-10-00-00 +0000,wsj,"Election Day is here, with control of the Hous...",https://www.wsj.com/articles/midterms-2026-ele...
74,53,2026-11-03-10-02-48 +0000,bbc,Jet may have broken apart before Florida crash...,https://www.bbc.co.uk/news/articles/cm9wz01dx8...
197,44,2026-11-02-23-12-59 +0000,chitri,Supreme Court rejects Trump administration pus...,https://www.chicagotribune.com/2026/11/02/supr...
35,41,2026-11-03-11-00-00 +0000,nypost,An hour-by-hour guide to watching the 2026 mid...,https://nypost.com/2026/11/03/us-news/midterms...
395,40,2026-11-02-11-59-01 +0000,nypost,Hackers breach emergency alert apps in Ukraine...,https://nypost.com/2026/11/02/world-news/hacke...
186,39,2026-11-02-23-51-48 +0000,nypost,Michigan Dems join Haley Stevens in abandoning...,https://nypost.com/2026/11/02/us-news/michigan...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
